# Walkie Checkpoint Utility

源码导航：[`core/utils/walkie_checkpoint.py`](../../../core/utils/walkie_checkpoint.py) 中的 `unwrap_model`、`save_walkie_checkpoint`、`load_walkie_checkpoint`、`apply_walkie_checkpoint`、`resolve_resume_path`。

训练中断后重新恢复要求不止于权重。若仅保存 `model.state_dict()`，则优化器动量缓存、学习率调度器位置和随机数状态均丢失，导致恢复后的训练轨迹偏移。Walkie 的 checkpoint 工具将所有必要状态打包为统一格式，训练脚本仅需调用单一接口即可完整恢复。

### 1. 理论背景：完整训练状态的定义

一次完整的训练 checkpoint $\mathcal{C}$ 应包含：

$$
\mathcal{C} = \{\,\theta,\; S_{\text{opt}},\; S_{\text{scaler}},\; S_{\text{schedule}},\; t,\; \text{stage},\; S_{\text{rng}},\; \text{cfg}\,\}
$$

| 字段 | 含义 |
|---|---|
| $\theta$ | 模型参数 `model.state_dict()` |
| $S_{\text{opt}}$ | 各优化器状态（AdamW 的 $m, v$；Muon 的牛顿-舒尔茨矩 等） |
| $S_{\text{scaler}}$ | GradScaler 状态（AMP loss scale 历史） |
| $S_{\text{schedule}}$ | WSD 调度器状态（step, stage, warmup/decay 进度） |
| $t$ | 当前全局 step |
| $\text{stage}$ | 训练阶段标识（`warmup` / `main` / `decay`） |
| $S_{\text{rng}}$ | Python / NumPy / torch / CUDA 随机数状态 |
| $\text{cfg}` | 序列化的模型与训练配置快照（用于架构校验） |

### 2. 落盘格式与路径约定

```
runs/walkie/<run_name>/
    latest.pt         # 最新 step 的完整快照（每步覆盖）
    best.pt           # 最优 eval metric 对应的快照
    step_00010000.pt  # 可选的周期性存档（由 format='step' 触发）
```

写入时先写临时文件 `*.pt.tmp`，再 `os.replace()` 原子替换，防止程序崩溃时写入损坏。

### 3. 架构一致性校验

`load_walkie_checkpoint` 在加载时与传入的 `expected_model_cfg` 对比以下关键字段：

`vocab_size, n_layer, n_embd, n_head, n_head_kv, head_dim, d_ffn, tie_weights`

任意字段不一致时抛出 `RuntimeError`，防止加载架构不匹配的 checkpoint 到错误的模型上。

In [ ]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.model.walkie import WalkieConfig, WalkieForCausalLM
from core.utils.walkie_checkpoint import (
    apply_walkie_checkpoint,
    load_walkie_checkpoint,
    resolve_resume_path,
    save_walkie_checkpoint,
    unwrap_model,
)

### 4. 保存与恢复 tiny 模型的端到端验证

In [ ]:
import tempfile

# 构造极小模型供测试
cfg = WalkieConfig(
    vocab_size=64, block_size=32, n_embd=32,
    n_layer=1, n_head=4, n_head_kv=2, head_dim=8, d_ffn=64,
)
model     = WalkieForCausalLM(cfg)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

with tempfile.TemporaryDirectory() as tmp:
    out_dir = Path(tmp)

    # --- 保存 ---
    path = save_walkie_checkpoint(
        out_dir,
        model=model,
        optimizers={'adamw': optimizer},
        scaler=None,
        schedule_state={'step': 3, 'stage': 'warmup'},
        step=3,
        stage='warmup',
        best_metric=1.23,
        model_cfg=cfg.to_dict(),
        train_cfg={'batch_size': 2},
        format='latest',
    )
    print("saved  :", path.name)

    # resolve_resume_path 优先返回 latest.pt
    print("resolved:", resolve_resume_path(out_dir).name)

    # --- 加载并应用 ---
    payload  = load_walkie_checkpoint(path, expected_model_cfg=cfg.to_dict())
    restored = WalkieForCausalLM(cfg)
    info     = apply_walkie_checkpoint(payload, model=restored, optimizers=None, scaler=None)
    print("step    :", info['step'])
    print("stage   :", info['stage'])
    print("missing :", info['missing'])   # 应为空列表
    print("unexpected:", info['unexpected'])

### 5. DDP / torch.compile 包装剥离验证

In [ ]:
import torch.nn as nn

# 模拟 torch.compile 包装（原模型放在 _orig_mod 属性上）
class FakeCompileWrapper(nn.Module):
    def __init__(self, inner):
        super().__init__()
        self._orig_mod = inner

wrapped = FakeCompileWrapper(model)
unwrapped = unwrap_model(wrapped)

print("unwrapped is model:", unwrapped is model)  # 应为 True
assert unwrapped is model, "剥离 torch.compile 包装失败！"

# 模拟 DDP 包装（原模型放在 .module 属性上）
class FakeDDPWrapper(nn.Module):
    def __init__(self, inner):
        super().__init__()
        self.module = inner

ddp_wrapped = FakeDDPWrapper(model)
assert unwrap_model(ddp_wrapped) is model
print("DDP 剥离也正确。")

### 6. 源码精讲

**`save_walkie_checkpoint` 的原子写入**：

```python
payload = {
    "version": WALKIE_CKPT_VERSION,
    "model":      unwrap_model(model).state_dict(),  # 剥离 DDP/compile 后保存
    "optimizers": {k: opt.state_dict() for k, opt in optimizers.items()},
    "scaler":     scaler.state_dict() if scaler else None,
    "schedule":   schedule_state,      # 调度器状态（含 WSD 三段进度）
    "step":       int(step),
    "stage":      stage,
    "rng_state":  _collect_rng_state(),  # Python/NumPy/torch/CUDA 随机数状态
    "model_cfg":  model_cfg,   # 配置快照，用于加载时的架构校验
    "train_cfg":  train_cfg,
}
tmp = path.with_suffix(path.suffix + ".tmp")
torch.save(payload, tmp)
os.replace(tmp, path)   # 原子替换，防止中途崩溃写出损坏文件
```

**`unwrap_model` 的双层剥离**：

```python
def unwrap_model(model):
    inner = model
    # 剥离 DDP（.module）
    if hasattr(inner, "module") and isinstance(inner.module, nn.Module):
        inner = inner.module
    # 剥离 torch.compile（._orig_mod）
    if hasattr(inner, "_orig_mod") and isinstance(inner._orig_mod, nn.Module):
        inner = inner._orig_mod
    return inner
```

注意顺序：先剥 DDP 再剥 compile，与 PyTorch 的包装嵌套顺序一致。

---

## 延伸阅读与参考资料

### PyTorch 官方文档
- **Saving and Loading Models**: [tutorial](https://pytorch.org/tutorials/beginner/saving_loading_models.html)
- **Automatic Mixed Precision examples**: [docs](https://pytorch.org/docs/stable/notes/amp_examples.html)

### 工程实践
- **torch.save / torch.load serialization notes**: [docs](https://pytorch.org/docs/stable/notes/serialization.html)
- **Distributed Checkpoint (FSDP)**: [docs](https://pytorch.org/docs/stable/fsdp.html)